# Module 1 — Take-Home Exercises: Strategy & Data Selection

These exercises reinforce what you learned in class about dataset selection,
data quality auditing, and the go/no-go decision for fine-tuning.

**No GPU required** for Exercises 1-3. Exercise 4 needs a T4 GPU (Colab free tier).

---

## Exercise 1: Evaluate a New Dataset (Conceptual)

You're building a **mental health chatbot** for a university counseling center.
A colleague suggests fine-tuning on the `Amod/mental_health_counseling_conversations`
dataset from Hugging Face.

**Tasks:**

1. Load the dataset and inspect 10 random examples
2. Check for the same quality issues we found in ChatDoctor:
   - Persona contamination (therapist names, platform references)
   - Boilerplate greetings/sign-offs
   - Safety disclaimers (crisis hotline numbers, "seek professional help")
   - Very short responses (<50 characters)
3. Write a 3-sentence assessment: would you fine-tune on this dataset? Why or why not?

In [ ]:
!pip install -q datasets pandas

In [ ]:
from datasets import load_dataset
import re

# TODO: Load the mental health dataset
# ds = load_dataset("Amod/mental_health_counseling_conversations", split="train")

# TODO: Print dataset size and column names

# TODO: Display 10 random examples

In [ ]:
# TODO: Audit for quality issues
# Adapt the audit code from the in-class exercise to check:
#   1. Persona contamination patterns (look for therapist names, platform refs)
#   2. Boilerplate greetings/sign-offs
#   3. Safety disclaimers (crisis hotline, "seek professional help", "call 988")
#   4. Very short responses (<50 chars)

# HINT: First read 10-20 examples manually to identify what patterns exist,
#       then write regex patterns to quantify them.

**Your Assessment (fill in):**

1. Dataset quality: (good / mixed / poor)
2. Key issue found: ...
3. Would you fine-tune on this? Why or why not?

---

## Exercise 2: Dataset Size vs Quality Trade-off

In class, we learned that 1,000 high-quality examples can beat 10,000 noisy ones.

Given the ChatDoctor dataset (112K examples, 63.1% persona contamination),
answer these questions:

**Tasks:**

1. If you filter out ALL examples with persona contamination, how many remain?
2. If you also filter out examples with boilerplate sign-offs (28.2%), and assume
   some overlap with persona contamination, estimate the remaining clean examples.
3. Would this filtered dataset be better for fine-tuning than the full 112K? Why?
4. Write a filtering function that removes contaminated examples. Run it and
   report the actual count.

In [ ]:
from datasets import load_dataset
import re

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
print(f"Original size: {len(ds):,}")

# Persona contamination patterns (from the in-class exercise)
PERSONA_PATTERNS = [
    r"chat\s*doctor", r"welcome to", r"thanks for",
    r"posting (?:your|the) query", r"i understand your concern"
]

BOILERPLATE_PATTERNS = [
    r"hope this helps", r"wishing you", r"good luck",
    r"wish you good health", r"take care"
]

# TODO: Write a function that returns True if an example is CLEAN
# (no persona contamination AND no boilerplate)
def is_clean(example):
    output = example["output"].lower()
    # Check persona patterns
    # Check boilerplate patterns
    # Return True only if NONE of the patterns match
    pass

# TODO: Filter the dataset
# clean_ds = ds.filter(is_clean)
# print(f"Clean examples: {len(clean_ds):,} ({len(clean_ds)/len(ds)*100:.1f}%)")

**Your Answers (fill in):**

1. Examples remaining after removing persona contamination: ___
2. Estimated clean examples (no persona, no boilerplate): ___
3. Better for fine-tuning? (yes/no and why): ...
4. Actual count from your filter: ___

---

## Exercise 3: Design Your Own Reformatting Prompt

In class, we used `data_prep_v2.py` to reformat WikiDoc via GPT-4o-mini.
The prompt told GPT-4o-mini to convert terse encyclopedia text into
conversational healthcare-assistant responses.

**Scenario:** You want to fine-tune a model for a **pediatrics helpline**
that helps parents understand common childhood illnesses.

**Tasks:**

1. Write a system prompt for GPT-4o-mini that would reformat medical text
   into parent-friendly answers. Consider:
   - Tone (reassuring but not dismissive)
   - Reading level (non-medical parents)
   - When to say "call your pediatrician" vs "go to the ER now"
   - Safety disclaimers appropriate for worried parents
2. Test your prompt on the sample input below by calling the OpenAI API
   (or write what you think the output should look like)

In [ ]:
# Sample raw medical text (like WikiDoc)
SAMPLE_INPUT = """
Question: What causes croup in children?
Answer: Croup is most commonly caused by parainfluenza viruses, particularly 
types 1 and 3. Other causes include influenza A and B, adenovirus, respiratory 
syncytial virus (RSV), and measles. The infection causes swelling of the larynx, 
trachea, and bronchi. Peak incidence is between 6 months and 3 years of age, 
with most cases occurring in autumn.
"""

# TODO: Write your reformatting prompt
REFORMATTING_PROMPT = """
You are reformatting medical text into parent-friendly answers for a
pediatrics helpline chatbot.

[YOUR INSTRUCTIONS HERE — consider:
  - What tone should the answer use?
  - What reading level?
  - What safety disclaimers?
  - How should it handle urgency levels?]
"""

# TODO (optional): If you have an OpenAI API key, test it:
# from openai import OpenAI
# client = OpenAI(api_key="sk-...")
# response = client.chat.completions.create(
#     model="gpt-4o-mini",
#     messages=[
#         {"role": "system", "content": REFORMATTING_PROMPT},
#         {"role": "user", "content": SAMPLE_INPUT}
#     ]
# )
# print(response.choices[0].message.content)

---

## Exercise 4: System Prompt Ablation Study (T4 GPU required)

In class, we tested 3 system prompts. Now test **2 more** to see how far
prompt engineering can go.

**Tasks:**

1. Design a "safety-first" system prompt that prioritizes disclaimers above all else
2. Design a "structured" system prompt that demands bullet points, headers, and numbered lists
3. Run both on the same 10 benchmark questions
4. Compare: which prompt produces the most helpful output? The safest?

In [ ]:
!pip install -q transformers torch bitsandbytes accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")

BENCHMARK_PROMPTS = [
    "What are the common symptoms of Type 2 diabetes?",
    "How does hypertension affect the heart over time?",
    "What is the recommended first-line treatment for mild persistent asthma?",
    "Explain the difference between Type 1 and Type 2 diabetes.",
    "What are the common side effects of ibuprofen?",
    "How is pneumonia typically diagnosed?",
    "What lifestyle changes help manage high cholesterol?",
    "What are the early warning signs of a stroke?",
    "How does metformin work for diabetes management?",
    "What vaccinations are recommended for adults over 65?",
]

def generate_response(prompt, system_prompt, model, tokenizer, max_new_tokens=300):
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7, top_p=0.9, do_sample=True)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# TODO: Design your two system prompts

SAFETY_FIRST_PROMPT = """
[YOUR PROMPT — prioritize safety disclaimers, "consult a doctor" messaging,
 crisis numbers, and "this is not medical advice" above all else]
"""

STRUCTURED_PROMPT = """
[YOUR PROMPT — demand bullet points, headers, numbered lists,
 clear section structure, and organized formatting]
"""

# Run on 3 test questions first to iterate quickly
test_questions = BENCHMARK_PROMPTS[:3]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"\n--- SAFETY-FIRST ---")
    print(generate_response(q, SAFETY_FIRST_PROMPT, model, tokenizer))
    print(f"\n--- STRUCTURED ---")
    print(generate_response(q, STRUCTURED_PROMPT, model, tokenizer))

In [ ]:
# TODO: After reviewing the 3-question test, run the full 10-question benchmark
# with your best prompt. Save results for comparison.

# results = []
# for q in BENCHMARK_PROMPTS:
#     results.append({
#         "question": q,
#         "safety_first": generate_response(q, SAFETY_FIRST_PROMPT, model, tokenizer),
#         "structured": generate_response(q, STRUCTURED_PROMPT, model, tokenizer),
#     })

**Your Findings (fill in):**

1. Which prompt produced more helpful answers? Why?
2. Which prompt produced safer answers? Why?
3. Could any of these prompts match what fine-tuning achieved in Module 2?
4. What does this tell you about when to use prompt engineering vs fine-tuning?